<a href="https://colab.research.google.com/github/shakhaoathpappu-jpg/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shakhaoathpappu-jpg/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# SETUP — discover what's actually in the warehouse before assuming any column names
!pip install -q huggingface_hub pandas pyarrow scikit-learn

from huggingface_hub import HfApi, hf_hub_download
from google.colab import userdata
import pandas as pd

HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID = "FlyRank/internship-warehouse"

api = HfApi()
files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN)
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Config used throughout this notebook — write your text answer in the markdown cell above, not here.
LANE = "content_refresh_prioritization"
MID_PANEL_MONTH = "2026-03"
PRIOR_MONTH = "2026-02"       # needed for the trend-direction feature
SEALED_TEST_MONTH = "2026-06" # never touch this for label logic


## Unit of analysis + time window

One row represents one webpage for one monthly observation.

I will use the Content Refresh Prioritization lane.

The analysis will use a mid-panel month (2026-03) to avoid using the final month as the outcome window.

The goal is to rank webpages that are most likely to need refreshing based on their search performance.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields: feature / label / context / excluded

**Features**
- Clicks
- Impressions
- CTR
- Average Position
- Trend Direction

**Label / Proxy**
- Refresh Priority Score (or ranking priority)

**Context**
- Month
- Page

**Excluded**
- Future outcome information and any label-derived columns.

Reason: They would introduce data leakage and make the evaluation unrealistic.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# Load mid-panel month, prior month, and the content dimension table
mid_path = hf_hub_download(
    repo_id=REPO_ID, repo_type="dataset",
    filename=f"fact_content_daily_performance/month={MID_PANEL_MONTH}/data_0.parquet",
    token=HF_TOKEN
)
prior_path = hf_hub_download(
    repo_id=REPO_ID, repo_type="dataset",
    filename=f"fact_content_daily_performance/month={PRIOR_MONTH}/data_0.parquet",
    token=HF_TOKEN
)
dim_content_path = hf_hub_download(
    repo_id=REPO_ID, repo_type="dataset",
    filename="dim_content.parquet",
    token=HF_TOKEN
)

df_mid = pd.read_parquet(mid_path)
df_prior = pd.read_parquet(prior_path)
dim_content = pd.read_parquet(dim_content_path)

print("=== fact_content_daily_performance (mid-panel month) ===")
print("Shape:", df_mid.shape)
print("Columns:", df_mid.columns.tolist())
print(df_mid.dtypes)
print(df_mid.head())

print("\n=== dim_content ===")
print("Shape:", dim_content.shape)
print("Columns:", dim_content.columns.tolist())
print(dim_content.head())

=== fact_content_daily_performance (mid-panel month) ===
Shape: (9841378, 30)
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
report_date                  object
client_hash_id               object
content_hash_id              object
client_has_gsc                 bool
client_has_ga4                 bool
gsc_data_available             bool
ga4_data_available           object
gsc_impressions               int64
gsc_clicks                    int64
gsc_sum_position              int64
gsc_avg_position           

In [5]:
# QUERY 1 — GRAIN: raw table is one row per (page, day); we build page-month by aggregating
raw_dupes = df_mid.duplicated(subset=["client_hash_id", "content_hash_id", "report_date"]).sum()
print(f"Raw daily rows in {MID_PANEL_MONTH}: {len(df_mid)}")
print(f"Duplicate (client, content, report_date) rows: {raw_dupes}")
assert raw_dupes == 0, "Raw grain claim failed."

# Aggregate daily rows up to PAGE x MONTH — this is our actual unit of analysis
monthly = (
    df_mid.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_sum_position=("gsc_sum_position", "sum"),
        days_with_gsc=("gsc_data_available", "sum"),
        n_days=("report_date", "count"),
    )
)
monthly["month"] = MID_PANEL_MONTH
monthly["ctr"] = monthly["gsc_clicks"] / monthly["gsc_impressions"].replace(0, pd.NA)
monthly["avg_position"] = monthly["gsc_sum_position"] / monthly["gsc_impressions"].replace(0, pd.NA)

month_dupes = monthly.duplicated(subset=["client_hash_id", "content_hash_id"]).sum()
print(f"Aggregated page-month rows: {len(monthly)}")
print(f"Duplicate (page) rows after aggregation: {month_dupes}")
assert month_dupes == 0, "Page-month grain failed after aggregation."

Raw daily rows in 2026-03: 9841378
Duplicate (client, content, report_date) rows: 0
Aggregated page-month rows: 331437
Duplicate (page) rows after aggregation: 0


In [6]:
# QUERY 2 — ROW COUNT + DATE SPAN
print("Raw daily rows loaded for mid-panel month:", len(df_mid))
print("Report date span:", df_mid["report_date"].min(), "to", df_mid["report_date"].max())
print("Aggregated page-month rows:", len(monthly))
print("Unique clients:", monthly["client_hash_id"].nunique())
print("Unique pages:", monthly["content_hash_id"].nunique())

Raw daily rows loaded for mid-panel month: 9841378
Report date span: 2026-03-01 to 2026-03-31
Aggregated page-month rows: 331437
Unique clients: 55
Unique pages: 331437


In [7]:
# QUERY 3 — AVAILABILITY: filter with IS TRUE, show survivors
raw_available = df_mid[df_mid["gsc_data_available"] == True]
available = monthly[monthly["days_with_gsc"] > 0].copy()

print(f"Raw rows before filter: {len(df_mid)}")
print(f"Raw rows where gsc_data_available IS TRUE: {len(raw_available)}")
print(f"Page-months before filter: {len(monthly)}")
print(f"Page-months with >=1 gsc_data_available day: {len(available)}")
dropped = len(monthly) - len(available)
print(f"Dropped: {dropped} page-months ({dropped/len(monthly):.1%})")

Raw rows before filter: 9841378
Raw rows where gsc_data_available IS TRUE: 3611061
Page-months before filter: 331437
Page-months with >=1 gsc_data_available day: 176738
Dropped: 154699 page-months (46.7%)


### Five features (max) — each with an "available when?" line

1. **Clicks** — knowable at the decision moment because it's the closed count of clicks recorded through the end of the decision month.
2. **Impressions** — knowable at the decision moment for the same reason: it's a closed monthly total, not a forward-looking number.
3. **CTR** — knowable because it's simply clicks ÷ impressions, both already closed for the month.
4. **Average Position** — knowable because Search Console reports it as a completed monthly average, not a live/future estimate.
5. **Trend Direction** — knowable because it only compares the decision month to the *prior* month, never to a future one.

In [8]:
# Aggregate the PRIOR month the same way, for the trend feature
monthly_prior = (
    df_prior.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(prior_clicks=("gsc_clicks", "sum"))
)

features_df = available.merge(monthly_prior, on=["client_hash_id", "content_hash_id"], how="left")

def trend(row):
    if pd.isna(row["prior_clicks"]):
        return "no_prior_data"
    diff = row["gsc_clicks"] - row["prior_clicks"]
    return "up" if diff > 0 else ("down" if diff < 0 else "flat")

features_df["trend_direction"] = features_df.apply(trend, axis=1)

features_df = features_df[[
    "client_hash_id", "content_hash_id", "month",
    "gsc_clicks", "gsc_impressions", "ctr", "avg_position", "trend_direction", "prior_clicks"
]]

features_df.head()

,client_hash_id,content_hash_id,month,gsc_clicks,gsc_impressions,ctr,avg_position,trend_direction,prior_clicks
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,2026-03,0,1,0.0,9.0,flat,0.0
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03,2,331,0.006042,14.377644,up,1.0
2,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,2026-03,0,33,0.0,9.363636,flat,0.0
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,2026-03,0,145,0.0,8.124138,flat,0.0
4,client_0797ff3a1fc9a6a5,content_1207efddce873942,2026-03,0,461,0.0,14.488069,flat,0.0


### The trap — add ONE label-derived column on purpose

We define a simple proxy label ("needs refresh" = clicks fell month-over-month), train an honest model, then sneak in a column that is basically the label restated as a feature, and watch the score jump toward perfect. Then we delete it.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

features_df["needs_refresh"] = (features_df["trend_direction"] == "down").astype(int)
y = features_df["needs_refresh"]

X_honest = features_df[["gsc_clicks", "gsc_impressions", "ctr", "avg_position"]].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
honest_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])
print(f"Honest AUC (no leakage): {honest_auc:.3f}")

X_leaky = X_honest.copy()
X_leaky["clicks_delta"] = features_df["gsc_clicks"] - features_df["prior_clicks"].fillna(0)

Xl_train, Xl_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
leaky_model = LogisticRegression(max_iter=1000).fit(Xl_train, y_train)
leaky_auc = roc_auc_score(y_test, leaky_model.predict_proba(Xl_test)[:, 1])
print(f"Leaky AUC (with 'clicks_delta'): {leaky_auc:.3f}")

print(f"\nScore jumped from {honest_auc:.3f} to {leaky_auc:.3f} just by adding a column that encodes the label.")
del X_leaky
print(f"Leaky column removed. The honest AUC ({honest_auc:.3f}) is the number we keep.")

/tmp/ipykernel_10084/903989244.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_honest = features_df[["gsc_clicks", "gsc_impressions", "ctr", "avg_position"]].fillna(0)


Honest AUC (no leakage): 0.643
Leaky AUC (with 'clicks_delta'): 1.000

Score jumped from 0.643 to 1.000 just by adding a column that encodes the label.
Leaky column removed. The honest AUC (0.643) is the number we keep.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

- This dataset cannot prove causal relationships — it shows association between search signals and refresh need, not proof that refreshing *causes* better performance.
- The analysis provides decision support based on observed search performance, not a guarantee.
- Historical data may be incomplete for some pages (e.g. newly published pages have fewer prior months to compute a trend from), and future search behavior can't be guaranteed from past observations.
- **Named limitation of this slice:** pages with fewer than one prior month of history can't get a `trend_direction` feature at all, so this lane's ranking is systematically less reliable for newly published or newly indexed pages.
- Results should be read as directional/decision-support, not as absolute predictions.
The raw fact table is daily-grain; the page-month unit of analysis is built by aggregating gsc_clicks/gsc_impressions within each month

In [10]:
print("Missing values in the feature frame:")
print(features_df.isnull().sum())

no_prior_month = features_df["trend_direction"].eq("no_prior_data").sum()
print(f"\nPage-months with no prior-month data (can't compute a real trend): {no_prior_month}")

Missing values in the feature frame:
client_hash_id         0
content_hash_id        0
month                  0
gsc_clicks             0
gsc_impressions        0
ctr                    0
avg_position           0
trend_direction        0
prior_clicks       15199
needs_refresh          0
dtype: int64

Page-months with no prior-month data (can't compute a real trend): 15199


## Data limits

This dataset cannot prove causal relationships.

The analysis provides decision support based on observed search performance.

Historical data may be incomplete, and future search behavior cannot be guaranteed from past observations.

The results should be interpreted as directional rather than absolute predictions.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.